# Stage 2 — Exploratory Data Analysis
## Shopee Singapore Listing Samples (18 August 2025)

**Scope note:** every chart in this notebook describes the **sampled** Category/Keyword/Store listings in this dataset only — six personal-care/household Category groups, seventeen related Keyword search terms, and twelve Store handles, collected in brief crawl bursts on 2025-08-18. None of these charts should be read as describing the Singapore e-commerce marketplace broadly. `sold`/`monthly_sold`/`rating`/`review` fields are source-reported platform counters, not independently audited sales, revenue, or customer-satisfaction measures. See `docs/STAGE1_DATASET_REVIEW.md` and `docs/STAGE2_DATA_CLEANING.md` for full provenance and cleaning detail.

Inputs: `data/processed/listings_observations.csv` (2,996 cleaned observation rows) and `data/processed/listings_unique_reference.csv` (2,727 distinct listing keys), produced by `scripts/clean_data.py` from the original raw workbooks."


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

obs = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "listings_observations.csv",
    dtype={"shop_id": str, "product_id": str, "id": str},
    parse_dates=["crawl_time", "parse_time"],
)
unique_listings = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "listings_unique_reference.csv",
    dtype={"shop_id": str, "product_id": str},
)

print(f"Observations: {len(obs)} | Distinct listing keys: {unique_listings.shape[0]}")
obs["source_view"].value_counts()

## Chart 1 — Listing-price distribution and outliers

Using the 2,727 distinct listing keys (not the 2,996 observations, to avoid a listing counted in two views inflating the distribution) and each listing's minimum recorded price across views. Prices are historical listing-price observations from the 18 August 2025 crawl, not completed transaction prices.

In [ ]:
price = unique_listings["price_min_across_views"].dropna()
q1, q3 = price.quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
n_outliers = int((price > upper_fence).sum())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [2, 1]})

axes[0].hist(price, bins=60, color="#3b6fa0", edgecolor="white", linewidth=0.3)
axes[0].set_title("Listing price distribution (2,727 sampled listings, SGD)")
axes[0].set_xlabel("Price (SGD)")
axes[0].set_ylabel("Number of listings")
axes[0].axvline(price.median(), color="#c0392b", linestyle="--", linewidth=1,
                 label=f"Median SGD {price.median():.2f}")
axes[0].legend(frameon=False)

axes[1].boxplot(price, vert=True, widths=0.5, patch_artist=True,
                 boxprops=dict(facecolor="#3b6fa0", alpha=0.5),
                 medianprops=dict(color="#c0392b"))
axes[1].set_title(f"Outliers above SGD {upper_fence:.0f}: n={n_outliers}")
axes[1].set_xticks([])
axes[1].set_ylabel("Price (SGD)")

fig.suptitle("Sampled Shopee SG listings only — not the full marketplace", fontsize=8, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"n={len(price)} | min={price.min():.2f} | median={price.median():.2f} | "
      f"max={price.max():.2f} | IQR upper fence={upper_fence:.2f} | outliers={n_outliers}")

## Chart 2 — Price comparison across the six sampled Category groups

Category view only (998 observations; this view has no repeated shop/product pairs feeding into this specific chart beyond the 42 noted in Stage 1 — each repeat is a separate row here since category membership is itself the thing being compared). Variant/pack-size price ranges are not reconciled in this chart; see `variant_comparability_uncertain` in the cleaned data for listings where `min_price` != `max_price`.

In [ ]:
cat_obs = obs[obs["source_view"] == "category"].copy()
order = cat_obs.groupby("category_name")["price"].median().sort_values().index

fig, ax = plt.subplots(figsize=(9, 4.5))
data_by_cat = [cat_obs.loc[cat_obs["category_name"] == c, "price"].dropna() for c in order]
bp = ax.boxplot(data_by_cat, vert=False, widths=0.6, patch_artist=True,
                 boxprops=dict(facecolor="#3b6fa0", alpha=0.6),
                 medianprops=dict(color="#c0392b"))
ax.set_yticks(range(1, len(order) + 1))
counts = cat_obs["category_name"].value_counts()
ax.set_yticklabels([f"{c} (n={counts[c]})" for c in order])
ax.set_xlabel("Price (SGD)")
ax.set_title("Price by Category group — Category view, 6 sampled groups")

plt.tight_layout()
plt.savefig(FIG_DIR / "02_price_by_category.png", dpi=150, bbox_inches="tight")
plt.show()

cat_obs.groupby("category_name")["price"].median().sort_values()

## Chart 3 — Recorded-discount availability and consistency by view

`discount_consistency_status` compares the recorded `discount` field against a price-implied discount (`1 − price/price_before_discount`), computed only where the reference price is valid (excludes the 312 Store `-1` sentinel rows). A matching stated and implied percentage is **not** proof of a genuine promotion — variant/pack-size comparability is not confirmed here (see Chart 2 note and `variant_comparability_uncertain`).

In [ ]:
status_order = ["consistent", "inconsistent", "no_recorded_discount",
                "invalid_or_missing_reference_price", "no_recorded_and_no_valid_reference"]
status_labels = ["Consistent", "Inconsistent (>1pt)", "No recorded discount",
                 "Invalid/missing reference price", "No recorded + no valid reference"]
colors = ["#2e7d32", "#c0392b", "#9e9e9e", "#e0a800", "#5d4037"]

pivot = (
    obs.groupby(["source_view", "discount_consistency_status"]).size()
    .unstack(fill_value=0)
    .reindex(columns=status_order, fill_value=0)
    .reindex(["category", "keyword", "store"])
)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8.5, 4.5))
left = pd.Series(0, index=pivot_pct.index, dtype=float)
for status, label, color in zip(status_order, status_labels, colors):
    ax.barh(pivot_pct.index, pivot_pct[status], left=left, color=color, label=label)
    left += pivot_pct[status]

ax.set_xlabel("Share of identifiable observations (%)")
ax.set_title("Discount field status by collection view")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_discount_status_by_view.png", dpi=150, bbox_inches="tight")
plt.show()

pivot

## Chart 4 — Price-implied discount distribution (eligible observations)

Restricted to rows with a valid reference price (`reference_price_status == "valid"`) and a computable implied discount — this is the `1 − price/price_before_discount` figure, independent of whatever the `discount` field states. Shown separately from the recorded discount per the project brief; the two should not be assumed equivalent for any individual listing (see Chart 3).

In [ ]:
implied = obs.loc[obs["reference_price_status"] == "valid", "discount_implied_pct"].dropna()
implied = implied[(implied >= 0) & (implied <= 100)]  # drop non-sensical implied values if any

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(implied, bins=50, color="#6a51a3", edgecolor="white", linewidth=0.3)
ax.axvline(implied.median(), color="#c0392b", linestyle="--", linewidth=1,
           label=f"Median {implied.median():.1f}%")
ax.set_xlabel("Price-implied discount (%)")
ax.set_ylabel("Number of observations")
ax.set_title(f"Price-implied discount — eligible observations (n={len(implied)})")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_implied_discount_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

implied.describe()

## Chart 5 — Recorded rating vs. review count

Both fields are source-reported platform counters (not independently validated). Shown per distinct listing (first observation) to avoid double-counting listings that appear in multiple views. Log-scaled review count axis since review counts are highly right-skewed.

In [ ]:
one_per_listing = obs.sort_values("source_excel_row").drop_duplicates("listing_key", keep="first")
plot_df = one_per_listing[["rating", "reviews_count"]].dropna()
plot_df = plot_df[plot_df["reviews_count"] > 0]  # log scale requires > 0

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_df["reviews_count"], plot_df["rating"], alpha=0.25, s=14, color="#3b6fa0")
ax.set_xscale("log")
ax.set_xlabel("Recorded review count (log scale)")
ax.set_ylabel("Recorded rating")
ax.set_title(f"Rating vs. review count — one row per distinct listing (n={len(plot_df)})")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_rating_vs_reviews.png", dpi=150, bbox_inches="tight")
plt.show()

plot_df.describe()

## Chart 6 — Seller coverage and assortment limitation

The Store view samples only **12 distinct shop handles** (999 listings). This chart shows how unevenly those 999 listings are spread across the 12 shops, to make the shallow seller sample explicit before any seller-level comparison is attempted — see the Stage 1 finding that seller-level conclusions should stay descriptive, not representative.

In [ ]:
store_obs = obs[obs["source_view"] == "store"].copy()
store_counts = store_obs["store_name"].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(store_counts.index, store_counts.values, color="#3b6fa0")
ax.set_xlabel("Number of listings sampled (Store view)")
ax.set_title(f"Listings per sampled store handle (n={len(store_counts)} stores, 999 listings)")
for y, v in enumerate(store_counts.values):
    ax.text(v + 2, y, str(v), va="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_store_assortment.png", dpi=150, bbox_inches="tight")
plt.show()

store_counts.describe()

## Chart 7 — Sponsored vs. non-sponsored composition, Keyword view

`sponsored` is a source-provided flag on search results; it does not imply anything about listing quality or performance. Shown per the 17 sampled search keywords.

In [ ]:
kw_obs = obs[obs["source_view"] == "keyword"].copy()
sponsor_pivot = (
    kw_obs.groupby(["search_keyword", "sponsored"]).size().unstack(fill_value=0)
    .rename(columns={True: "Sponsored", False: "Organic"})
)
sponsor_pivot = sponsor_pivot.loc[sponsor_pivot.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.barh(sponsor_pivot.index, sponsor_pivot["Organic"], color="#9e9e9e", label="Organic")
ax.barh(sponsor_pivot.index, sponsor_pivot["Sponsored"], left=sponsor_pivot["Organic"],
        color="#e0a800", label="Sponsored")
ax.set_xlabel("Number of observations")
ax.set_title("Sponsored vs. organic observations per sampled keyword")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "07_sponsored_by_keyword.png", dpi=150, bbox_inches="tight")
plt.show()

kw_obs["sponsored"].value_counts(normalize=True) * 100

## Summary

See `docs/STAGE2_EDA_FINDINGS.md` for findings written up with verified output statistics and caveats. In brief: prices and ratings are usable for descriptive comparison within the sampled categories/keywords/stores; the recorded `discount` field requires row-level validation against price fields before use (chart 3); the Store view's 12-shop seller sample is too shallow for seller-strategy claims (chart 6); and none of `sold`, `monthly_sold`, `rating`, or `reviews_count` should be read as audited sales, revenue, or verified satisfaction.